<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>Sharif University of Technology</b></p>
    <p>Deep Learning Course, Dr. Soleymani</p>
    <p>Spring 2026</p>
</div>

---


*Full Name:* Amirhosein Rezaei

*Student ID:* 401105989

# Retrieval-Augmented Generation (RAG) – From Scratch to Conversation

## Overview

**Retrieval-Augmented Generation (RAG)** is a technique that enhances a language model’s output by first retrieving relevant information from a knowledge base, then conditioning the generation on that retrieved evidence.

Mathematically, given a query $x$, we want to model:

$
p(y \mid x) = \sum_{z \in \text{Top-}k(x)} p_{\text{retrieve}}(z \mid x) \; p_{\text{generate}}(y \mid x, z)
$

where $z$ is a retrieved document chunk, often treated as a hard selection after top‑k retrieval.

### Dataset used in this notebook

Instead of a manually written toy knowledge base, this version uses a small real subset of **SQuAD v1** from Hugging Face. SQuAD provides real Wikipedia-based contexts, questions, and gold answers. To keep the notebook lightweight, we only use a small validation subset.

### In this notebook you will:

1. **From scratch (no LangChain)**:
   - Load a small real QA dataset
   - Chunk real Wikipedia contexts
   - Create dense embeddings with SentenceTransformers
   - Build a FAISS index for fast nearest-neighbour search
   - Implement a retriever and a GPT‑2 generator
   - Assemble a complete, stateless RAG system

2. **With LangChain**:
   - Integrate memory for multi-turn conversations
   - Implement a **history-aware** retriever that rewrites follow-up questions

3. **With an encoder-decoder generator**:
   - Replace GPT‑2 with FLAN‑T5
   - Compare decoder-only RAG and encoder-decoder RAG on the same retrieved contexts

After completing this notebook, you will understand the core components of RAG, the need for memory in conversational systems, and the architectural difference between decoder-only and encoder-decoder generators.

**Dependencies**: Run the cell below to install required packages.

In [ ]:
# Install necessary packages (if not already installed)
import sys
!{sys.executable} -m pip install --quiet torch transformers sentence-transformers faiss-cpu datasets langchain-community langchain-huggingface langchain-core
# We use:
# - datasets for loading a small real SQuAD subset
# - sentence-transformers for dense embeddings
# - faiss-cpu for vector search
# - transformers for GPT-2 and FLAN-T5 generators
# - langchain-community / langchain-huggingface for the vectorstore wrapper in the conversational part


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Part 1: Document Loading & Text Chunking

A RAG system needs a collection of documents to retrieve from. In real applications these are often large texts, so we must split them into manageable **chunks**. Each chunk is a unit that can be embedded and later retrieved.

In this notebook, we use a small real subset of **SQuAD v1** instead of a manually written toy text file. Each SQuAD example contains:

- a real Wikipedia paragraph as `context`,
- a `question`,
- one or more gold `answers`.

We use the contexts as the retrieval corpus and the questions/gold answers for testing.

**Why chunk?**
- Language models have a limited context window.
- Smaller chunks allow more precise retrieval; a long document may contain many topics.

**Chunking trade‑offs**:
- Too small → loss of surrounding context.
- Too large → retrieval may bring irrelevant detail, and generation may exceed token limits.

**Implementation**: We’ll implement a simple recursive character‑based splitter with user‑defined chunk size and overlap.

### Questions 1
1. What would happen if we used whole documents without chunking?

would exceed model context window and retrieval would bring irrelevant information and reducing precision and long documents contain multiple topics so query might match wrong section and slower retrieval and generation

2. How does the overlap parameter help preserve continuity between chunks?

prevents information loss at chunk boundaries and ensures entities sentences split across boundaries are captured in multiple chunks and maintains context flow between consecutive chunks

In [ ]:
# Implement `chunk_text`.
# Split text into overlapping character chunks and return a list of strings.
# Expected output is kept below as a reference.

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """
    Split text into overlapping chunks of approximately chunk_size characters.
    Overlap ensures continuity between chunks.
    """
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end]
        chunks.append(chunk)
        if end == text_len:
            break
        start += chunk_size - overlap
    return chunks

# test small example
test_text = "This is a test sentence. " * 20
test_chunks = chunk_text(test_text, chunk_size=100, overlap=20)
print(f"Number of chunks: {len(test_chunks)}")
print(f"First chunk length: {len(test_chunks[0])}")

Number of chunks: 6
First chunk length: 100


In [ ]:
# Load a small real SQuAD subset using `load_dataset`.
# Then extract unique contexts and create `chunks` using `chunk_text`.
# Expected output is kept below as a reference.

import requests
import json
from datasets import Dataset

url = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json"
response = requests.get(url)
data = response.json()

samples = []
for article in data["data"]:
    title = article["title"]
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            if not qa["answers"]:
                continue
            samples.append({
                "id": qa["id"],
                "title": title,
                "context": context,
                "question": qa["question"],
                "answers": {"text": [ans["text"] for ans in qa["answers"]],
                            "answer_start": [ans["answer_start"] for ans in qa["answers"]]}
            })

qa_dataset = Dataset.from_list(samples[:300])

print(qa_dataset)
print("Example row:")
print(qa_dataset[0])

unique_contexts = {}
for example in qa_dataset:
    ctx = example["context"]
    if ctx not in unique_contexts:
        unique_contexts[ctx] = example["title"]

print(f"Number of QA examples: {len(qa_dataset)}")
print(f"Number of unique contexts: {len(unique_contexts)}")

all_chunks = []
chunk_to_source = []
for ctx, title in unique_contexts.items():
    chunks = chunk_text(ctx, chunk_size=500, overlap=50)
    all_chunks.extend(chunks)
    chunk_to_source.extend([title] * len(chunks))

print(f"Created {len(all_chunks)} chunks from real SQuAD contexts.\n")

for i in range(min(3, len(all_chunks))):
    print(f"Chunk {i}:")
    print(all_chunks[i][:300] + " ...\n")

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})
Example row:
{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represented

## Part 2: Document Embeddings & Vector Index

To retrieve relevant chunks, we need to embed both the documents and the query into a common dense vector space. We use a pre‑trained sentence transformer to obtain **fixed‑size embeddings**.

**Dense retrieval** computes:

$
\text{sim}(q, d) = \cos(\phi(q), \phi(d)) \quad \text{or} \quad \|\phi(q) - \phi(d)\|_2
$

where $\phi$ is the embedding function.

We’ll store the embeddings in a **FAISS** index for fast approximate (or exact) nearest‑neighbour search.

### Questions 2
1. Explain the difference between sparse retrieval (e.g., TF‑IDF, BM25) and dense retrieval.

in sparse we have bag of words and term matching with xact keyword matches	and high dimensional and sparse vectors	and don't need for training but in dense we have semantic meaning and also handles synonyms and paraphrases and have low dimensional and dense vectors and requires training and pretrained modles

2. Why is L2 distance often used instead of cosine similarity in FAISS `IndexFlatL2`? (Hint: embedding normalisation)

for normalized embeddings like L2 norm = 1 the L2 distance is equivalent to cosine ||a-b||^2 = 2(1 - cos(a,b)) so L2 is faster in FAISS and works directly with normalized vectors

In [ ]:
# Load `all-MiniLM-L6-v2` with SentenceTransformer.
# Encode all chunks into float32 dense embeddings.
# Expected output is kept below as a reference.

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embed_model.to(device)
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

chunk_embeddings = embed_model.encode(all_chunks, convert_to_numpy=True, show_progress_bar=True)
print(f"Embeddings shape: {chunk_embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_5607/3296938659.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (30, 384)


In [ ]:
# Build a FAISS IndexFlatL2 index.
# Add all chunk embeddings to the index.
# Expected output is kept below as a reference.

import faiss

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype(np.float32))
print(f"FAISS index contains {index.ntotal} vectors.")

FAISS index contains 30 vectors.


## Part 3: The Retriever

The retriever is responsible for, given a user query, finding the top‑k most relevant chunks.

Formally:

$
\text{Retrieve}(q, k) = \text{argtopk}_{d \in \mathcal{D}} \; \text{sim}(\phi(q), \phi(d))
$

where $\mathcal{D}$ is the set of all chunk embeddings.

### Question 3
How would you modify the retriever to use **maximum inner product search** (MIPS) instead of L2 distance? When would MIPS be preferred?

with replace IndexFlatL2 with IndexFlatIP and then normalize embeddings before adding. when embeddings are trained with dot product objective like CLIP or some sentence transformers or when you need maximum inner product search for recommendation systems

In [ ]:
# Implement `retrieve(query, index, embed_model, chunks, top_k)`.
# Embed the query, search FAISS, and return top-k (chunk, distance) pairs.
# Expected output is kept below as a reference.

def retrieve(query: str, index: faiss.Index, embed_model: SentenceTransformer,
             chunks: List[str], top_k: int = 3) -> List[Tuple[str, float]]:
    """
    Retrieve top_k chunks most similar to the query.
    Returns list of (chunk_text, distance) pairs.
    """
    query_emb = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(query_emb, top_k)
    results = [(chunks[idx], float(distances[0][i])) for i, idx in enumerate(indices[0])]
    return results

sample_question = qa_dataset[0]["question"]
gold_answer = qa_dataset[0]["answers"]["text"][0]
retrieved = retrieve(sample_question, index, embed_model, all_chunks, top_k=3)

print(f"Question: {sample_question}")
print(f"Gold answer: {gold_answer}\n")
print("Top retrieved chunks:\n")
for i, (chunk, dist) in enumerate(retrieved):
    print(f"Distance {dist:.4f}:")
    print(chunk[:200] + " ...\n")


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Top retrieved chunks:

Distance 0.5626:
feating them 20–18 in the AFC Championship Game. They joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl. ...

Distance 0.6448:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated ...

Distance 0.8009:
The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NFC Championship  ...



## Part 4: The Generator (GPT‑2)

We use a pre‑trained auto‑regressive language model (GPT‑2) to generate an answer conditioned on the retrieved context and the question.

**Input format** we will use:

Context:

chunk 1

chunk 2
...

Question: {user query}

Answer:

GPT‑2 will then continue the string.

### Questions 4
1. Why do we concatenate the context and the question?

GPT-2 is decoder only so needs all information in single input sequence and enables model to attend to context while generating answer and is simpler than encoder decoder cross attention

2. What are the limitations of using a base GPT‑2 model for knowledge‑intensive generation?

small context window and not instruction tuned so continues prompt instead of answering and prone to hallucination and not built in "I don't know" capability and also poor at extraction tasks compared to encoder decoder models

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_model_name = "gpt2"  # You can also try "gpt2-medium" if resources allow.
tokenizer = GPT2Tokenizer.from_pretrained(gpt2_model_name)
model = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
model.to(device)
model.eval()

# GPT-2 has no pad token by default; set it to eos for generation.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Implement `build_rag_prompt` and `generate_answer` for GPT-2.
# Concatenate retrieved context and question, then generate an answer.
# Expected output is kept below as a reference.

def build_rag_prompt(retrieved_chunks: List[str], question: str) -> str:
    context = "\n".join(retrieved_chunks)
    prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    return prompt

def generate_answer(prompt: str, max_new_tokens: int = 50) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in generated:
        after_answer = generated.split("Answer:", 1)[1].strip()
        for stop_char in ["\n", ".", "Question:"]:
            if stop_char in after_answer:
                after_answer = after_answer.split(stop_char)[0]
                break
        return after_answer.strip() + "."

    answer = generated[len(prompt):].strip()
    if "Question:" in answer:
        answer = answer.split("Question:")[0].strip()
    return answer if answer else "I don't know"

# Test
retrieved_chunks = retrieve(sample_question, index, embed_model, all_chunks, top_k=3)
retrieved_texts = [chunk for chunk, _ in retrieved_chunks]
test_prompt = build_rag_prompt(retrieved_texts, sample_question)
test_answer = generate_answer(test_prompt, max_new_tokens=30)
print(f"Question: {sample_question}")
print(f"Gold answer: {gold_answer}")
print(f"GPT-2 answer: {test_answer}")


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos
GPT-2 answer: The New England Patriots..


## Part 5: Putting It All Together – Stateless RAG

Now we combine the retriever and the generator into a single pipeline.

Given a question:
1. Retrieve top‑k chunks.
2. Concatenate them into a context string.
3. Feed context + question to the GPT‑2 generator.

Let’s implement a `RAGSystem` class that encapsulates this process.

### Question 5
This system answers each question independently. What problem arises when a user asks a follow‑up question like *“Tell me more about that”* or *“Multiply the previous answer by 2”*?

it lack explicit references and retriever can't resolve "that game" without memory of previous topic and each query is independent so loses conversation context and Cannot handle coreference resolution like it or that or there

In [ ]:
# Implement the stateless `RAGSystem` class.
# It should retrieve chunks, build the context, and generate an answer.
# Expected output is kept below as a reference.

class RAGSystem:
    def __init__(self, index: faiss.Index, embed_model: SentenceTransformer,
                 chunks: List[str], generator_model, tokenizer, device):
        self.index = index
        self.embed_model = embed_model
        self.chunks = chunks
        self.model = generator_model
        self.tokenizer = tokenizer
        self.device = device

    def query(self, question: str, top_k: int = 3, max_new_tokens: int = 80) -> Dict:
        retrieved = retrieve(question, self.index, self.embed_model, self.chunks, top_k)
        retrieved_texts = [chunk for chunk, _ in retrieved]
        prompt = build_rag_prompt(retrieved_texts, question)
        answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
        return {
            "question": question,
            "answer": answer,
            "retrieved_chunks": retrieved_texts,
            "distances": [dist for _, dist in retrieved]
        }

rag = RAGSystem(index, embed_model, all_chunks, model, tokenizer, device)

# Test
result = rag.query(sample_question, top_k=3, max_new_tokens=80)
print(f"Question: {result['question']}")
print(f"Gold answer: {gold_answer}\n")
print("Retrieved chunks:")
for chunk in result['retrieved_chunks']:
    print(f"- {chunk[:150]}...")
print(f"\nGPT-2 RAG answer: {result['answer']}")

Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Retrieved chunks:
- feating them 20–18 in the AFC Championship Game. They joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made...
- Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football...
- The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated th...

GPT-2 RAG answer: The New England Patriots..


## Part 6: The Memory Problem

Let’s simulate a conversation where the second question depends on the first one.

**Stateless RAG will fail or become unstable** because each call is independent. The retriever only sees the latest question, so a follow-up such as *“Where was it played?”* or *“Which team represented the NFC in that game?”* is ambiguous unless the system remembers that the previous topic was **Super Bowl 50**.


In [ ]:
q1 = qa_dataset[0]["question"]
ans1 = rag.query(q1, top_k=3, max_new_tokens=80)
print(f"Q: {q1}\nA: {ans1['answer']}\n")

# Follow-up that refers to the previous topic.
# Without memory, the phrase "that game" is ambiguous for the retriever.
q2 = "Which team represented the NFC in that game?"
ans2 = rag.query(q2, top_k=3, max_new_tokens=80)
print(f"Q: {q2}\nA: {ans2['answer']}\n")

# Notice: the second answer may be weaker because stateless RAG does not explicitly
# remember that the previous question was about Super Bowl 50.

Q: Which NFL team represented the AFC at Super Bowl 50?
A: The New England Patriots..

Q: Which team represented the NFC in that game?
A: The New England Patriots..



## Part 7: Adding Memory with LangChain

To enable multi-turn conversations, we need to **maintain a chat history** and use it to reformulate the current question into a standalone query.

In many LangChain tutorials, this is done with functions such as `create_history_aware_retriever` and `create_retrieval_chain`. However, these imports can break across different LangChain versions. To keep this notebook stable, we implement the same logic explicitly while still using LangChain for the FAISS vectorstore wrapper.

We will:

1. Wrap the same FAISS index and embedding model with LangChain’s `FAISS` vectorstore.
2. Store the conversation in a simple `chat_history` list.
3. Rewrite ambiguous follow-up questions into standalone questions.
4. Retrieve documents using the standalone question.
5. Generate the final answer using **FLAN-T5**, which is much better for instruction following than base GPT-2.

### Questions 6
1. Explain how a history-aware retriever works internally. What prompt or rewriting step does it use?

takes chat history with current question as input and uses an LLM to rewrite question into standalone form and as example "Where was it played?" with history it will be "Where was Super Bowl 50 played?" and standalone query then goes to retriever

2. Why can’t we simply pass the whole chat history to the retriever instead of generating a standalone question?

chat history contains irrelevant words so reduces retrieval precision and vector search works best with short and focused queries and also history length grows over time so may exceed model limits and standalone query preserves necessary context while being concise

In [ ]:
# Create LangChain `Document` objects from chunks.
# Then build a FAISS vectorstore using HuggingFace embeddings.
# Expected output is kept below as a reference.

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

langchain_docs = [Document(page_content=chunk) for chunk in all_chunks]

hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(langchain_docs, hf_embeddings)
print(f"Vectorstore contains {vectorstore.index.ntotal} documents.")

/tmp/ipykernel_5607/2337708122.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore contains 30 documents.


In [ ]:
# Load `google/flan-t5-small` as an encoder-decoder model.
# Implement a small helper function to generate text with FLAN-T5.
# Expected output is kept below as a reference.

from transformers import T5Tokenizer, T5ForConditionalGeneration

flan_model_name = "google/flan-t5-small"
flan_tokenizer = T5Tokenizer.from_pretrained(flan_model_name)
flan_model = T5ForConditionalGeneration.from_pretrained(flan_model_name)
flan_model.to(device)
flan_model.eval()

def generate_with_t5(prompt: str, max_new_tokens: int = 100) -> str:
    """Generate answer using FLAN-T5."""
    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = flan_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    answer = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# Test
test_t5_prompt = "Answer the following question: What is the capital of France?"
print(generate_with_t5(test_t5_prompt))

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

l'île de l'île


In [ ]:
# Implement helper functions for conversational RAG.
# Convert chat history to text and rewrite short follow-up questions.

def format_chat_history(chat_history: List[Tuple[str, str]]) -> str:
    formatted = ""
    for role, text in chat_history:
        if role == "human":
            formatted += f"Human: {text}\n"
        else:
            formatted += f"Assistant: {text}\n"
    return formatted

def rewrite_question(question: str, chat_history: List[Tuple[str, str]]) -> str:
    if not chat_history:
        return question

    history_str = format_chat_history(chat_history)
    rewrite_prompt = f"""Given the following conversation between a human and an assistant, rewrite the last question to be a standalone, self-contained question that can be understood without the conversation history.

Conversation:
{history_str}
Human: {question}

Standalone question:"""

    standalone = generate_with_t5(rewrite_prompt, max_new_tokens=50)
    standalone = standalone.split("\n")[0].strip()
    return standalone if standalone else question

# Test
test_history = [("human", "What was Super Bowl 50?"), ("ai", "It was an American football game.")]
test_followup = "Where was it played?"
print("Original follow-up:", test_followup)
print("Rewritten:", rewrite_question(test_followup, test_history))

Original follow-up: Where was it played?
Rewritten: What was Super Bowl 50?


In [ ]:
# Implement `SimpleConversationalRAG`.
# Steps: rewrite follow-up question, retrieve documents, generate answer.

class SimpleConversationalRAG:
    def __init__(self, vectorstore, embed_model, llm_model, llm_tokenizer):
        self.vectorstore = vectorstore
        self.embed_model = embed_model
        self.llm_model = llm_model
        self.llm_tokenizer = llm_tokenizer

    def query(self, question: str, chat_history: List[Tuple[str, str]], top_k: int = 3) -> Dict:
        standalone_q = rewrite_question(question, chat_history)

        retrieved_docs = self.vectorstore.similarity_search(standalone_q, k=top_k)
        retrieved_texts = [doc.page_content for doc in retrieved_docs]

        context = "\n\n".join(retrieved_texts)
        final_prompt = f"""Answer the question based only on the following context. If the context does not contain the answer, say "I don't know".

Context:
{context}

Question: {standalone_q}

Answer:"""

        answer = generate_with_t5(final_prompt, max_new_tokens=100)

        return {
            "original_question": question,
            "standalone_question": standalone_q,
            "answer": answer,
            "retrieved_context": retrieved_texts
        }

conv_rag = SimpleConversationalRAG(vectorstore, hf_embeddings, flan_model, flan_tokenizer)

# Test
test_history = [("human", "What was Super Bowl 50?"), ("ai", "Super Bowl 50 was an American football game.")]
result = conv_rag.query("Where was it played?", test_history)
print("Standalone question:", result["standalone_question"])
print("Answer:", result["answer"])

Standalone question: What was Super Bowl 50?
Answer: an American football game


In [ ]:
# Test the corrected memory-based RAG pipeline.
rag_chain_with_memory = conv_rag

chat_history = []

question_1 = "What was Super Bowl 50?"
response_1 = rag_chain_with_memory.query(question_1, chat_history)

print("Question 1:", question_1)
print("Standalone question 1:", response_1["standalone_question"])
print("Answer 1:", response_1["answer"])

chat_history.append(("human", question_1))
chat_history.append(("ai", response_1["answer"]))

question_2 = "Where was it played?"
response_2 = rag_chain_with_memory.query(question_2, chat_history)

print("\nQuestion 2:", question_2)
print("Standalone question 2:", response_2["standalone_question"])
print("Answer 2:", response_2["answer"])
print("\nRetrieved context for turn 2:")
print(response_2["retrieved_context"][:1000], "...")


Question 1: What was Super Bowl 50?
Standalone question 1: What was Super Bowl 50?
Answer 1: an American football game

Question 2: Where was it played?
Standalone question 2: What was Super Bowl 50?
Answer 2: an American football game

Retrieved context for turn 2:
['Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv', 'In early 2012, NFL Commissioner Roger Goodell stated that the league planned to make the 50th Super Bowl "spectacular" and that it would be "an important game for us as a league".', 'CBS broadcast Super Bowl 50 in the U.S., an

In [ ]:
chat_history = []


def ask_question(query: str):
    response = conv_rag.query(query, chat_history)

    # Update chat history after each turn.
    chat_history.append(("human", query))
    chat_history.append(("ai", response["answer"]))
    return response


# First turn: a real SQuAD-style question.
q1 = qa_dataset[0]["question"]
response1 = ask_question(q1)
print("User:", q1)
print("Standalone:", response1["standalone_question"])
print("Assistant:", response1["answer"])

# Follow-up question that depends on the previous topic.
q2 = "Which team represented the NFC in that game?"
response2 = ask_question(q2)
print("\nUser:", q2)
print("Standalone:", response2["standalone_question"])
print("Assistant:", response2["answer"])


User: Which NFL team represented the AFC at Super Bowl 50?
Standalone: Which NFL team represented the AFC at Super Bowl 50?
Assistant: Denver Broncos

User: Which team represented the NFC in that game?
Standalone: Which team represented the AFC in Super Bowl 50?
Assistant: Denver Broncos


### Explanation for Question 6

A history-aware retriever first checks whether the latest user question is self-contained. If the question contains references such as *it*, *that game*, or *the previous answer*, the system uses the chat history to rewrite it into a standalone query. For example:

```text
Chat history: What was Super Bowl 50?
Follow-up: Where was it played?
Standalone query: Where was Super Bowl 50 played?
```

The retriever should usually receive this standalone query rather than the whole chat history. Passing the entire chat history directly to the retriever can introduce irrelevant words from previous turns and make vector search less focused. A short standalone query preserves the missing context while keeping retrieval precise.

In this notebook, we implement the history-aware logic explicitly instead of relying on `langchain.chains`, because those imports are version-sensitive. The idea is the same: use memory to rewrite the query, retrieve relevant documents, then answer from the retrieved context.


## Part 8: RAG with an Encoder-Decoder Generator

So far, the retriever has been encoder-based, but the generator has been GPT‑2, which is a decoder-only language model.

In this part, we replace GPT‑2 with an encoder-decoder model, **google/flan-t5-small**. The retrieved context and the user question are passed to the encoder, and the decoder generates the answer.

### Question 7
Implement a RAG pipeline that uses the same retriever as before, but replaces GPT‑2 with an encoder-decoder model such as `google/flan-t5-small`.

1. Compare the GPT‑2-based RAG and the T5-based RAG on at least two questions.

GPT-2 RAG produces poor and hallucinated answers and doesn't follow instructions and uses only left to right context attention and often continues the prompt instead of answering directly but T5 RAG produces good answers by extracting from context and follows instructions well and uses bidirectional encoder attention and outputs are clean with direct answers.

2. Explain how the retrieved context is used differently in GPT‑2 and T5.

in GPT-2 is concatenated as text prefix and causal attention only and in T5 it passed to encoder with bidirectional attention and decoder uses cross attention

3. Why is T5 considered an encoder-decoder model, while GPT‑2 is decoder-only?

T5 is separate encoder and processes input bidirectionally with decoder that generates autoregressively with cross attention to encoder but GPT-2 is only decoder stack and generates by predicting next token from previous tokens and not separate encoding stage

In [ ]:
# Implement the T5-based RAG pipeline.
# Use the same retriever, but replace GPT-2 with FLAN-T5.
# Expected output is kept below as a reference.

class T5RAGSystem:
    def __init__(self, index: faiss.Index, embed_model: SentenceTransformer,
                 chunks: List[str], t5_model, t5_tokenizer, device):
        self.index = index
        self.embed_model = embed_model
        self.chunks = chunks
        self.model = t5_model
        self.tokenizer = t5_tokenizer
        self.device = device

    def query(self, question: str, top_k: int = 3, max_new_tokens: int = 80) -> Dict:
        retrieved = retrieve(question, self.index, self.embed_model, self.chunks, top_k)
        retrieved_texts = [chunk for chunk, _ in retrieved]
        context = "\n\n".join(retrieved_texts)
        prompt = f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
        answer = generate_with_t5(prompt, max_new_tokens=max_new_tokens)
        return {
            "question": question,
            "answer": answer,
            "retrieved_chunks": retrieved_texts
        }

t5_rag = T5RAGSystem(index, embed_model, all_chunks, flan_model, flan_tokenizer, device)

# Test
t5_result = t5_rag.query(sample_question)
print(f"Question: {sample_question}")
print(f"Gold answer: {gold_answer}\n")
print("Retrieved context:")
print(t5_result["retrieved_chunks"][0][:300] + "...\n")
print("T5 answer:")
print(t5_result["answer"])

Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Retrieved context:
feating them 20–18 in the AFC Championship Game. They joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl....

T5 answer:
Denver Broncos


In [2]:
# Compare GPT-2 RAG and T5 RAG on at least two questions.
# Use the same retrieved context for both models for a fair comparison.
# Expected output is kept below as a reference.

def compare_on_question(question: str, gold_answer: str):
    retrieved = retrieve(question, index, embed_model, all_chunks, top_k=3)
    retrieved_texts = [chunk for chunk, _ in retrieved]

    prompt_gpt2 = build_rag_prompt(retrieved_texts, question)
    gpt2_answer = generate_answer(prompt_gpt2, max_new_tokens=80)

    context = "\n\n".join(retrieved_texts)
    prompt_t5 = f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
    t5_answer = generate_with_t5(prompt_t5, max_new_tokens=80)

    return {
        "question": question,
        "gold": gold_answer,
        "gpt2": gpt2_answer,
        "t5": t5_answer,
        "retrieved_context": retrieved_texts
    }

q1_data = qa_dataset[0]
q1 = q1_data["question"]
gold1 = q1_data["answers"]["text"][0]
comp1 = compare_on_question(q1, gold1)

q2_data = qa_dataset[1]
q2 = q2_data["question"]
gold2 = q2_data["answers"]["text"][0]
comp2 = compare_on_question(q2, gold2)

comparison_df = pd.DataFrame([
    {"Question": comp1["question"], "Gold Answer": comp1["gold"],
     "GPT-2 RAG Answer": comp1["gpt2"], "T5 Encoder-Decoder RAG Answer": comp1["t5"]},
    {"Question": comp2["question"], "Gold Answer": comp2["gold"],
     "GPT-2 RAG Answer": comp2["gpt2"], "T5 Encoder-Decoder RAG Answer": comp2["t5"]}
])
display(comparison_df)

print("="*100)
print(f"Question: {comp1['question']}")
print(f"Gold answer: {comp1['gold']}\n")
print("GPT-2 RAG Answer:")
print(comp1['gpt2'])
print("\nT5 Encoder-Decoder RAG Answer:")
print(comp1['t5'])
print("\nRetrieved Context:")
print(comp1['retrieved_context'][0][:500] + "...")
print("="*100)

print(f"\nQuestion: {comp2['question']}")
print(f"Gold answer: {comp2['gold']}\n")
print("GPT-2 RAG Answer:")
print(comp2['gpt2'])
print("\nT5 Encoder-Decoder RAG Answer:")
print(comp2['t5'])
print("\nRetrieved Context:")
print(comp2['retrieved_context'][0][:500] + "...")
print("="*100)

,Question,Gold Answer,GPT-2 RAG Answer,T5 Encoder-Decoder RAG Answer
0,Which NFL team represented the AFC at Super Bo...,Denver Broncos,The New England Patriots..,Denver Broncos
1,Which NFL team represented the NFC at Super Bo...,Carolina Panthers,The New York Giants..,Carolina Panthers


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

GPT-2 RAG Answer:
The New England Patriots..

T5 Encoder-Decoder RAG Answer:
Denver Broncos

Retrieved Context:
feating them 20–18 in the AFC Championship Game. They joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl....

Question: Which NFL team represented the NFC at Super Bowl 50?
Gold answer: Carolina Panthers

GPT-2 RAG Answer:
The New York Giants..

T5 Encoder-Decoder RAG Answer:
Carolina Panthers

Retrieved Context:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area

### Explanation for Question 7

In both systems, the retriever is exactly the same. The question is embedded using the SentenceTransformer model, and FAISS retrieves the top-k most relevant chunks from the real SQuAD context corpus.

The main difference is the generator architecture.

In the **GPT‑2-based RAG system**, the retrieved context, the question, and the `Answer:` prefix are concatenated into one prompt. GPT‑2 is a **decoder-only** model, so it generates the answer by continuing this prompt from left to right. During generation, each new token can attend only to previous tokens, including the retrieved context and the question.

In the **T5-based RAG system**, the retrieved context and the question are passed to the **encoder**. The encoder reads the full input sequence and produces contextual representations. Then the **decoder** generates the answer autoregressively while attending to the encoder outputs through cross-attention. Therefore, the retrieved context is used as an encoded source sequence rather than only as a text prefix.

T5 is considered an **encoder-decoder** model because it has two separate Transformer components: an encoder for processing the input text and a decoder for generating the output text. GPT‑2 is **decoder-only** because it only contains the autoregressive Transformer decoder stack and generates text by predicting the next token from the previous tokens.

In practice, FLAN‑T5 usually gives cleaner answers for this task than base GPT‑2 because FLAN‑T5 is instruction-tuned and better aligned with question answering. Base GPT‑2 is mainly trained for next-token prediction, so it may continue the prompt instead of directly answering the question.

## Summary

You have:
- Built a **stateless RAG system** from real SQuAD contexts, embeddings, FAISS, and GPT‑2.
- Observed its weakness on follow-up questions.
- Used a LangChain FAISS vectorstore and an explicit **history-aware query rewriting** step to add conversational memory.
- Replaced the unstable GPT‑2 memory generator with **FLAN‑T5** for reformulation and answering.
- Replaced the decoder-only GPT‑2 generator with an **encoder-decoder FLAN‑T5** generator.
- Compared GPT‑2-based RAG and T5-based RAG on the same retrieved contexts.

### Final Theoretical Questions
1. Compare the stateless and conversational RAG pipelines. What are the main differences in the retrieval step?

stateless: retrieves based only on current question and fails on follow ups like "that game"

conversational: rewrites question using chat history into a standalone query before retrieval and as example "Where was it played?" to "Where was Super Bowl 50 played?"

2. In the conversational version, where is the ‘memory’ actually stored?

in chat_history list as tuples of ("human", query) and ("ai", answer) and this list is passed to the rewrite_question function each turn

3. How could you extend this system to handle **multiple users**? What would you need to change?

create a session based dictionary mapping user_id to their chat_history and each user has independent memory also need user authentication

4. Suppose you wanted to replace GPT‑2 with a model that does not support the “reformulate a standalone question” task well. What alternatives could you explore?

use a smaller instruction tuned model like FLAN T5 and use rule based keyword replacement like replace "it" with last mentioned entity and use embedding similarity to find relevant past Q&A pairs and fine tune a small BERT model for query rewriting

5. Why can an encoder-decoder model such as T5 be more suitable for question answering than a base decoder-only model such as GPT‑2?

in T5  as encoder decoder the encoder processes full context bidirectionally and decoder generates answer with cross attention to encoded context to better at extraction and in GPT-2  as decoder only it is left to right only and trained for next token prediction and not instruction following to hallucinates or continues prompt instead of answering

### References
- [SQuAD: 100,000+ Questions for Machine Comprehension of Text](https://rajpurkar.github.io/SQuAD-explorer/)
- [Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (2020)](https://arxiv.org/abs/2005.11401)
- [FAISS library](https://github.com/facebookresearch/faiss)
- [FLAN-T5 model family](https://huggingface.co/google/flan-t5-small)
